# Round-8 #6 -- graph baselines at 5 matched seeds (per-seed false-safe counts)

Re-runs GCN, GraphSAGE, node-only GAT, and **ours** (GAT + edge fusion, coupled multi-task)
at seeds 0--4 on the grouped split, same recipe and zero-false-safe checkpoint rule. Reports
per-seed accuracy / macro-F1 / **false-safe count**, mean $\pm$ s.d., and pooled false-safe
(sum of counts over 5 seeds / 5$\times$Unstable). Saves `baselines_5seed_results.json`.
Runtime -> GPU, Run all.

In [ ]:
from google.colab import drive
import glob, os, sys
drive.mount('/content/drive')
c = glob.glob('/content/drive/MyDrive/**/smart_load_shield_boost', recursive=True)
FOLDER = c[0] if c else '/content/drive/MyDrive/smart_load_shield_boost'
sys.path.insert(0, FOLDER)
for f in ['contingency_data.npz', 'grouped_split.npz', 'boost_core.py']:
    assert os.path.exists(os.path.join(FOLDER, f)), 'missing ' + f
assert 'base_mask' in open(os.path.join(FOLDER, 'boost_core.py')).read(), 'boost_core.py not the masked version'
print('FOLDER =', FOLDER)

In [ ]:
import math, json, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import boost_core as B
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', DEV)
d = B.build_data(os.path.join(FOLDER, 'contingency_data.npz'), os.path.join(FOLDER, 'grouped_split.npz'), DEV)
SEEDS = [0, 1, 2, 3, 4]
HID, HEADS, NGAT, BS, LR, WD, LS, EPOCHS = 128, 4, 3, 1024, 3e-3, 1e-4, 0.05, 150
N_UNSTABLE = int((d['Tr'][d['idx_te']] == 2).sum())
print('test Unstable =', N_UNSTABLE)

class GCN(nn.Module):
    def __init__(s, di, do): super().__init__(); s.lin = nn.Linear(di, do)
    def forward(s, x, adj):
        deg = adj.sum(-1, keepdim=True).clamp(min=1); an = adj / deg.sqrt() / deg.transpose(1, 2).sqrt()
        return an @ s.lin(x)
class SAGE(nn.Module):
    def __init__(s, di, do): super().__init__(); s.lw = nn.Linear(di, do); s.nw = nn.Linear(di, do)
    def forward(s, x, adj):
        deg = adj.sum(-1, keepdim=True).clamp(min=1); nb = (adj @ x) / deg
        return s.lw(x) + s.nw(nb)
class SimpleGNN(nn.Module):
    def __init__(s, conv, node_dim=4, hidden=HID):
        super().__init__()
        s.enc = nn.Sequential(nn.Linear(node_dim, hidden), nn.GELU(), nn.LayerNorm(hidden))
        L = {'gat': lambda: B.GAT(hidden, hidden, HEADS, 0.15), 'gcn': lambda: GCN(hidden, hidden),
             'sage': lambda: SAGE(hidden, hidden)}[conv]
        s.convs = nn.ModuleList([L() for _ in range(NGAT)]); s.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(NGAT)])
        s.head = nn.Sequential(nn.Linear(hidden * 2, 64), nn.GELU(), nn.Dropout(0.15), nn.Linear(64, 3))
    def forward(s, nf, adj):
        h = s.enc(nf)
        for c, n in zip(s.convs, s.norms): h = n(h + F.gelu(c(h, adj)))
        return s.head(torch.cat([h.mean(1), h.max(1).values], -1))

@torch.no_grad()
def ev(net, idx, edge=False):
    net.eval(); P = []
    for s0 in range(0, len(idx), 4096):
        b = idx[s0:s0 + 4096]
        r = net(d['NF'][b], d['DENSE'][b], d['ADJ'][b])[0] if edge else net(d['NF'][b], d['ADJ'][b])
        P.append(r.argmax(1))
    P = torch.cat(P); R = d['Tr'][idx]
    acc = (P == R).float().mean().item()
    cm = torch.zeros(3, 3, dtype=torch.long)
    for t, p in zip(R, P): cm[t, p] += 1
    fs_n = int(cm[2, 0]); f1 = []
    for k in range(3):
        tp = cm[k, k].item(); fp = cm[:, k].sum().item() - tp; fn = cm[k].sum().item() - tp
        pr = tp / max(1, tp + fp); rc = tp / max(1, tp + fn); f1.append(2 * pr * rc / max(1e-9, pr + rc))
    return acc, fs_n, float(np.mean(f1))

In [ ]:
def train_baseline(conv, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    m = SimpleGNN(conv).to(DEV)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
    warm = max(1, EPOCHS // 30)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda e: min((e + 1) / warm, 1.0) * 0.5 * (1 + math.cos(math.pi * max(0, e - warm) / max(1, EPOCHS - warm))))
    ema = torch.optim.swa_utils.AveragedModel(m, multi_avg_fn=torch.optim.swa_utils.get_ema_multi_avg_fn(0.999))
    itr = d['idx_tr']; best, best_state = -1.0, None
    for ep in range(EPOCHS):
        m.train(); perm = itr[torch.randperm(len(itr), device=DEV)]
        for s0 in range(0, len(perm), BS):
            b = perm[s0:s0 + BS]
            loss = F.cross_entropy(m(d['NF'][b], d['ADJ'][b]), d['Tr'][b], weight=d['cw'], label_smoothing=LS)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(), 2.0)
            opt.step(); ema.update_parameters(m)
        sch.step()
        if ep >= EPOCHS // 3 and (ep % 5 == 0 or ep == EPOCHS - 1):
            for cand in (m, ema.module):
                a, fs, _ = ev(cand, d['idx_va'])
                if fs == 0 and a > best:
                    best = a; best_state = {k: v.clone() for k, v in cand.state_dict().items()}
    if best_state is None:
        best_state = max(((ev(c, d['idx_va'])[0], c) for c in (m, ema.module)), key=lambda t: t[0])[1].state_dict()
        best_state = {k: v.clone() for k, v in best_state.items()}
    ema.module.load_state_dict(best_state)
    return ev(ema.module, d['idx_te'])

def agg(name, rows):   # rows = list of (acc, fs_n, f1)
    A = np.array([r[0] for r in rows]); Fn = [r[1] for r in rows]; F1 = np.array([r[2] for r in rows])
    return dict(name=name, acc_mean=float(A.mean()), acc_std=float(A.std()),
                f1_mean=float(F1.mean()), f1_std=float(F1.std()),
                fs_per_seed=Fn, fs_pooled=int(sum(Fn)), pooled_unstable=N_UNSTABLE * len(rows),
                acc_all=[float(a) for a in A])

results = {}
# ---- ours: GAT + edge fusion, coupled multi-task (deployed config) ----
ours = []
for sd in SEEDS:
    te, model = B.train_eval(d, seed=sd, alpha=1.0, branched=False, stack=False, vbus=False, epochs=EPOCHS, amp=(DEV == 'cuda'))
    ours.append(ev(model, d['idx_te'], edge=True))
    print('ours   seed%d  acc %.2f%%  fs %d/%d  F1 %.2f%%' % (sd, ours[-1][0] * 100, ours[-1][1], N_UNSTABLE, ours[-1][2] * 100))
results['ours_gat_edge'] = agg('GAT + edge fusion (ours)', ours)
# ---- three graph baselines ----
for conv in ('sage', 'gcn', 'gat'):
    rows = []
    for sd in SEEDS:
        rows.append(train_baseline(conv, sd))
        print('%-5s seed%d  acc %.2f%%  fs %d/%d  F1 %.2f%%' % (conv, sd, rows[-1][0] * 100, rows[-1][1], N_UNSTABLE, rows[-1][2] * 100))
    results[conv] = agg(conv.upper(), rows)

In [ ]:
print('\n=== Table IV (5 seeds) ===')
print('%-26s %-16s %-14s %s' % ('architecture', 'acc mean+-sd', 'macro-F1', 'false-safe pooled'))
for k in ('ours_gat_edge', 'sage', 'gcn', 'gat'):
    r = results[k]
    print('%-26s %5.2f+-%.2f%%     %5.2f+-%.2f%%   %d / %d   per-seed %s'
          % (r['name'], r['acc_mean']*100, r['acc_std']*100, r['f1_mean']*100, r['f1_std']*100,
             r['fs_pooled'], r['pooled_unstable'], r['fs_per_seed']))
json.dump(results, open(os.path.join(FOLDER, 'baselines_5seed_results.json'), 'w'), indent=2)
print('\nsaved baselines_5seed_results.json -- download into redesign/')